# CSCI E-25
## Monocular depth estimation
### Steve Elston



## Introduction

Depth estimation has been a core CV task for decades, with numerous applications. The goal is to generate a depth or disparity (inverse of depth) map given one or more input images. The input images generally use an RGB representation, but can be gray scale.      

There are two common approaches to depth estimation. 
- Depth can be estimated from a pair of stereo images collected from calibrated cameras. This approach can provide unambiguous estimates of disparity. However, stereo depth estimation requires a calibrated pair of cameras in a fixed array.
- Alternatively, depth can be estimated from a single camera view or single pose, monocular depth estimation. This approach has the advantages of simplicity and can generate approximately correct depth maps. However, monocular depth estimation is inherently ambiguous and may produce incorrect results. Despite the disadvantage, we the will focus on the monocular approach here, given the simplicity.


In this notebook a multi-scale, encoder-decoder neural net is trained to estimate depth from an RGB image. Specifically, a UNET architecture is used to learn depth estimation from Dense Indoor and Outdoor Depth Dataset.    



### Attributions

[This notebook is based on a notebook in Keras Examples](https://keras.io/examples/vision/depth_estimation/) by [Victor Basu](https://www.linkedin.com/in/victor-basu-520958147) on 2024/08/13. Significant changes to the code and text discussion have been made.    

New code and numerous bug fixes in this notebook were generated in part using Google Gemini. 

## Key references

Key references for the algorithms used in this notebook and a few of the many possible extensions can be found in these papers.
1. [Depth Prediction Without the Sensors: Leveraging Structure for Unsupervised Learning from Monocular Videos](https://arxiv.org/abs/1811.06152v1), Casser, et. al., 2018. 
2. [Digging Into Self-Supervised Monocular Depth Estimation](https://openaccess.thecvf.com/content_ICCV_2019/papers/Godard_Digging_Into_Self-Supervised_Monocular_Depth_Estimation_ICCV_2019_paper.pdf), Godard, et. al., 2018. 
3. [Deeper Depth Prediction with Fully Convolutional Residual Networks](https://arxiv.org/abs/1606.00373v2), Laina, et. al., 2016. 


## Setup

To get started, execute the code in the cell below to import the required packages.   

In [ ]:
import os

os.environ["KERAS_BACKEND"] = "tensorflow"

import sys

import tensorflow as tf
import keras
from keras import layers
from keras import ops
import pandas as pd
import numpy as np
import cv2
import matplotlib.pyplot as plt

keras.utils.set_random_seed(123)

### Downloading the dataset

This notebook uses the [**DIODE: A Dense Indoor and Outdoor Depth**](https://diode-dataset.org/) dataset. DIODE contains images in RGB-D format. This format contains the a 3-channel RGB image of the scene. The fourth, D, channel contains the depth map. DIODE depth maps are created using multiple sensor scans. For indoor scenes an absolute relative accuracy, absREL, of 0.33 is claimed.    

The DIODE training dataset is 81GB. Consequently, we will use the validation dataset, which is 2.6GB. To further reduce the volume 

Other datasets other similar labeled depth estimation datasets you may wish to try include **[NYU-v2](https://cs.nyu.edu/~silberman/datasets/nyu_depth_v2.html)** and **[KITTI](http://www.cvlibs.net/datasets/kitti/)**.

Execute the code in the cell below to download the dataset.   

In [ ]:
annotation_folder = "/dataset/" # This variable was part of the original notebook, keeping it for context but it's not used in the download logic below.

# Define the actual paths for the downloaded archive and the extracted directory
download_file_path = os.path.join(os.path.abspath("."), "val.tar.gz")
extracted_dir_path = os.path.join(os.path.abspath("."), "val.tar.gz_extracted")

# Variable to hold the final path to the extracted data
annotation_zip = None

if os.path.exists(extracted_dir_path):
    # Case 1: Extracted directory already exists
    print(f"Extracted dataset found at: {extracted_dir_path}")
    print("No need to download or unzip. Proceeding with existing data.")
    annotation_zip = extracted_dir_path
elif os.path.exists(download_file_path):
    # Case 2: Zip file exists, but not the extracted directory
    print(f"Zip file already downloaded at: {download_file_path}")
    print("Proceeding with extraction.")
    # Call get_file to extract the existing zip. It will find the zip in cache_subdir
    # and extract it, returning the path to the extracted directory.
    annotation_zip = keras.utils.get_file(
        "val.tar.gz",
        cache_subdir=os.path.abspath("."),
        origin="http://diode-dataset.s3.amazonaws.com/val.tar.gz", # Origin is still needed for get_file to work even if file exists locally
        extract=True,
    )
else:
    # Case 3: Neither zip nor extracted directory exists, perform full download and extraction
    print("Dataset not found. Downloading and extracting DIODE validation dataset...")
    annotation_zip = keras.utils.get_file(
        "val.tar.gz",
        cache_subdir=os.path.abspath("."),
        origin="http://diode-dataset.s3.amazonaws.com/val.tar.gz",
        extract=True,
    )

# The annotation_zip variable now correctly holds the path to the extracted dataset.

###  Preparing the dataset

With the dataset downloaded, execute the code in the cell below to create a data frame containing the paths to the files containing the RGB image, the depth map and depth mask in the columns.  

In [ ]:
path = os.path.join(annotation_zip, "val/indoors")

filelist = []

for root, dirs, files in os.walk(path):
    for file in files:
        filelist.append(os.path.join(root, file))

filelist.sort()
data = {
    "image": [x for x in filelist if x.endswith(".png")],
    "depth": [x for x in filelist if x.endswith("_depth.npy")],
    "mask": [x for x in filelist if x.endswith("_depth_mask.npy")],
}
df = pd.DataFrame(data)

df = df.sample(frac=1, random_state=42)

### Setting the hyperparameters

Execute the code in the cell below to set some of the model hyperparameters.   

In [ ]:
HEIGHT = 256
WIDTH = 256
LR = 1e-4 # 1e-4
weight_decay = 0.2
EPOCHS = 60 #30
BATCH_SIZE = 32

## Building a data pipeline

The data pipeline reads and returns the data a batch at a time. Clean-up is performed at the end of each training epoch. This pipeline performs the following operations.   

1. The pipeline uses dataframe containing the paths for the RGB images, the depth and depth mask files.
2. The RGB images are read and resized.
3. The depth and depth mask files are read and processed to generate the resized depth map image.


Execute the code in the cell to load the DataGenerator object.  

In [ ]:
class DataGenerator(keras.utils.PyDataset):
    def __init__(self, data, batch_size=6, dim=(768, 1024), n_channels=3, shuffle=True):
        super().__init__()
        """
        Initialization
        """
        self.data = data
        self.indices = self.data.index.tolist()
        self.dim = dim
        self.n_channels = n_channels
        self.batch_size = batch_size
        self.shuffle = shuffle
        self.min_depth = 0.1
        self.on_epoch_end()

    def __len__(self):
        return int(np.ceil(len(self.data) / self.batch_size))

    def __getitem__(self, index):
        if (index + 1) * self.batch_size > len(self.indices):
            self.batch_size = len(self.indices) - index * self.batch_size
        # Generate one batch of data
        # Generate indices of the batch
        index = self.indices[index * self.batch_size : (index + 1) * self.batch_size]
        # Find list of IDs
        batch = [self.indices[k] for k in index]
        x, y = self.data_generation(batch)

        return x, y

    def on_epoch_end(self):
        """
        Updates indexes after each epoch
        """
        self.index = np.arange(len(self.indices))
        if self.shuffle == True:
            np.random.shuffle(self.index)

    def load(self, image_path, depth_map_path, mask_path):
        """Load input and target image."""

        image_ = cv2.imread(image_path)
        image_ = cv2.cvtColor(image_, cv2.COLOR_BGR2RGB)
        image_ = cv2.resize(image_, self.dim)
        image_ = tf.image.convert_image_dtype(image_, tf.float32)

        depth_map = np.load(depth_map_path).squeeze()
        mask = np.load(mask_path)
        mask = mask > 0

        max_depth = min(300, np.percentile(depth_map, 99))
        depth_map = np.clip(depth_map, self.min_depth, max_depth)

        # Define the lower bound for log-depth targets as per the original clipping logic
        log_min_depth_val = np.log(self.min_depth) # Corrected: Use np.log(self.min_depth)
        log_max_depth_val = np.log(max_depth)

        # Apply log transformation where mask is true, and fill with log_min_depth_val where mask is false
        # This ensures depth_map is a regular NumPy array without masked elements before resizing
        depth_map = np.where(mask, np.log(depth_map), log_min_depth_val)

        # Apply final clipping to the log-depths to ensure they are within the expected range
        depth_map = np.clip(depth_map, log_min_depth_val, log_max_depth_val)

        depth_map = cv2.resize(depth_map, self.dim)
        depth_map = np.expand_dims(depth_map, axis=2)
        depth_map = tf.image.convert_image_dtype(depth_map, tf.float32)

        return image_, depth_map

    def data_generation(self, batch):
        x = np.empty((self.batch_size, *self.dim, self.n_channels))
        y = np.empty((self.batch_size, *self.dim, 1))

        for i, batch_id in enumerate(batch):
            x[i,], y[i,] = self.load(
                self.data["image"][batch_id],
                self.data["depth"][batch_id],
                self.data["mask"][batch_id],
            )
        x, y = x.astype("float32"), y.astype("float32")
        return x, y

## Visualizing Data Samples

With the dataset ready and DataGenerator defined, you can now visualize some samples of the RGB images and the corresponding depth maps by executing the code in the cell below.   

In [ ]:
def visualize_depth_map(samples, test=False, model=None):
    input, target = samples
    cmap = plt.cm.jet
    cmap.set_bad(color="black")

    # Define a consistent vmin and vmax for depth maps (linear depth range)
    min_linear_depth = 0.1 # This was self.min_depth in DataGenerator
    max_depth = 10 # This was 300 in DataGenerator

    if test:
        pred_log = model.predict(input) # Model predicts log-depth
        #pred_linear = pred_log
        pred_linear = np.exp(pred_log) # Convert predicted to linear depth
        target_linear = np.exp(target) # Convert target to linear depth
        #target_linear = target #



        fig, ax = plt.subplots(6, 3, figsize=(50, 50))
        for i in range(6):
            max_linear_depth = min(max_depth, np.percentile(target_linear, 99))
            ax[i, 0].imshow((input[i].squeeze()))
            ax[i, 1].imshow((target_linear[i].squeeze()), cmap=cmap, vmin=min_linear_depth, vmax=max_linear_depth)

            max_linear_depth = min(max_depth, np.percentile(pred_linear, 99))
            ax[i, 2].imshow((pred_linear[i].squeeze()), cmap=cmap, vmin=min_linear_depth, vmax=max_linear_depth)

    else:
        target_linear = np.exp(target) # Convert target to log depth

        fig, ax = plt.subplots(6, 2, figsize=(50, 50))
        for i in range(6):
            max_linear_depth = min(max_depth, np.percentile(target_linear, 99))
            target_linear = np.exp(target) # Convert target to linear depth
            ax[i, 0].imshow((input[i].squeeze()))
            ax[i, 1].imshow((target_linear[i].squeeze()), cmap=cmap, vmin=min_linear_depth, vmax=max_linear_depth)

visualize_samples = next(
    iter(DataGenerator(data=df, batch_size=6, dim=(HEIGHT, WIDTH)))
)
visualize_depth_map(visualize_samples)

> **Exercise 11-1:** The RGB images of the scene are shown on the left and the corresponding depth map is show on the right. The depth-map color scale goes from dark blue for near-field objects to deep red for far-field objects. Carefully examine these image depth-map pairs. Notice the mirror in the background of the fourth image from the top; the image with the flower in a pot and a floor lamp. In one or two sentences, explain the anomaly in the depth map resulting from this reflective surface.  

> **Answer:**   

## 3D point cloud visualization

The target depth for each image was collected by using multiple scans of the scene. To get a feel for the raw 3-D point cloud results of these scans execute the code in the cell below to view an example.    

In [ ]:
depth_vis = np.flipud(visualize_samples[1][1].squeeze())  # target
img_vis = np.flipud(visualize_samples[0][1].squeeze())  # input

fig = plt.figure(figsize=(15, 10))
ax = plt.axes(projection="3d")

STEP = 3
for x in range(0, img_vis.shape[0], STEP):
    for y in range(0, img_vis.shape[1], STEP):
        ax.scatter(
            [depth_vis[x, y]] * 3,
            [y] * 3,
            [x] * 3,
            c=tuple(img_vis[x, y, :3] / 255),
            s=3,
        )
    ax.view_init(45, 135)

> **Exercise 11-2:** The point cloud shown above corresponds to the second image from the top of the auditorium. There are two brightly back-lit windows in the image. In one or two sentences explain the anomaly these windows create.       

> **Answer:**       

## Building the Model

We are now ready to construct the model. The model uses a UNET architecture. The classes for the block types used to construct the UNET model are defined in the code cell below.  

1. Down-sample block.    
2. Up-sample block.
3. Bottleneck block.

Execute the code in the cell below to load these classes.  

In [ ]:
class DownscaleBlock(layers.Layer):
    def __init__(
        self, filters, kernel_size=(3, 3), padding="same", strides=1, **kwargs
    ):
        super().__init__(**kwargs)
        drop_out_rate = 0.2
        self.convA = layers.Conv2D(filters, kernel_size, strides, padding)
        self.convB = layers.Conv2D(filters, kernel_size, strides, padding)
        self.reluA = layers.LeakyReLU(negative_slope=0.2)
        self.reluB = layers.LeakyReLU(negative_slope=0.2)
        self.bn2a = layers.BatchNormalization()
        self.bn2b = layers.BatchNormalization()
        self.doA = layers.Dropout(rate=drop_out_rate)
        self.doB = layers.Dropout(rate=drop_out_rate)

        self.pool = layers.MaxPool2D((2, 2), (2, 2))

    def call(self, input_tensor):
        d = self.convA(input_tensor)
        x = self.doA(d)
        x = self.bn2a(x)
        x = self.reluA(x)

        x = self.convB(x)
        x = self.doB(x)
        x = self.bn2b(x)
        x = self.reluB(x)

        x += d
        p = self.pool(x)
        return x, p


class UpscaleBlock(layers.Layer):
    def __init__(
        self, filters, kernel_size=(3, 3), padding="same", strides=1, **kwargs
    ):
        super().__init__(**kwargs)
        drop_out_rate = 0.2
        self.us = layers.UpSampling2D((2, 2))
        self.convA = layers.Conv2D(filters, kernel_size, strides, padding)
        self.convB = layers.Conv2D(filters, kernel_size, strides, padding)
        self.reluA = layers.LeakyReLU(negative_slope=0.2)
        self.reluB = layers.LeakyReLU(negative_slope=0.2)
        self.bn2a = layers.BatchNormalization()
        self.bn2b = layers.BatchNormalization()
        self.do2A = layers.Dropout(rate=drop_out_rate)
        self.do2B = layers.Dropout(rate=drop_out_rate)
        self.conc = layers.Concatenate()

    def call(self, x, skip):
        x = self.us(x)
        concat = self.conc([x, skip])
        x = self.convA(concat)
        x = self.do2A(x)
        x = self.bn2a(x)
        x = self.reluA(x)

        x = self.convB(x)
        x = self.do2B(x)
        x = self.bn2b(x)
        x = self.reluB(x)

        return x


class BottleNeckBlock(layers.Layer):
    def __init__(
        self, filters, kernel_size=(3, 3), padding="same", strides=1, **kwargs
    ):
        super().__init__(**kwargs)
        self.convA = layers.Conv2D(filters, kernel_size, strides, padding)
        self.convB = layers.Conv2D(filters, kernel_size, strides, padding)
        self.reluA = layers.LeakyReLU(negative_slope=0.2)
        self.reluB = layers.LeakyReLU(negative_slope=0.2)

    def call(self, x):
        x = self.convA(x)
        x = self.reluA(x)
        x = self.convB(x)
        x = self.reluB(x)
        return x

### Defining the model and loss function 

The code in the cell below constructs the UNET model with the following components.     

1. The DepthEstimationModel class with the `call` method that executes the UNET.    
2. The `calculate_loss` method computes the loss function with three components.     
   a. Structural similarity index(SSIM).    
   b. L1-loss, or Point-wise depth in our case.     
   c. Depth smoothness loss.      
3. The gradient, loss, loss components are computed by the `train_step` method. 
4. The loss and loss components are computed by the `test_step` method.    

Execute this code to load this class.  

In [ ]:
def image_gradients(image):
    if len(ops.shape(image)) != 4:
        raise ValueError(
            "image_gradients expects a 4D tensor "
            "[batch_size, h, w, d], not {}..".format(ops.shape(image))
        )

    image_shape = ops.shape(image)
    batch_size, height, width, depth = ops.unstack(image_shape)

    dy = image[:, 1:, :, :] - image[:, :-1, :, :]
    dx = image[:, :, 1:, :] - image[:, :, :-1, :]

    # Return tensors with same size as original image by concatenating
    # zeros. Place the gradient [I(x+1,y) - I(x,y)] on the base pixel (x, y).
    shape = ops.stack([batch_size, 1, width, depth])
    dy = ops.concatenate([dy, ops.zeros(shape, dtype=image.dtype)], axis=1)
    dy = ops.reshape(dy, image_shape)

    shape = ops.stack([batch_size, height, 1, depth])
    dx = ops.concatenate([dx, ops.zeros(shape, dtype=image.dtype)], axis=2)
    dx = ops.reshape(dx, image_shape)

    return dy, dx


class DepthEstimationModel(keras.Model):
    def __init__(self):
        super().__init__()

        ## Weight parameters for loss function
        self.ssim_loss_weight = 0.1 #0.5
        self.l1_loss_weight = 1.0
        self.edge_loss_weight = 0.5

        self.loss_metric = keras.metrics.Mean(name="loss")
        self.ssim_loss_metric = keras.metrics.Mean(name="ssim_loss")
        self.l1_loss_metric = keras.metrics.Mean(name="l1_loss")
        self.edge_loss_metric = keras.metrics.Mean(name="edge_loss")

        f = [16, 32, 64, 128, 256]
        self.downscale_blocks = [
            DownscaleBlock(f[0]),
            DownscaleBlock(f[1]),
            DownscaleBlock(f[2]),
            DownscaleBlock(f[3]),
        ]
        self.bottle_neck_block = BottleNeckBlock(f[4])
        self.upscale_blocks = [
            UpscaleBlock(f[3]),
            UpscaleBlock(f[2]),
            UpscaleBlock(f[1]),
            UpscaleBlock(f[0]),
        ]
        self.conv_layer = layers.Conv2D(1, (1, 1), padding="same", activation="linear")

    def calculate_loss(self, target, pred):
        # Edges
        dy_true, dx_true = image_gradients(target)
        dy_pred, dx_pred = image_gradients(pred)
        weights_x = ops.cast(ops.exp(ops.mean(ops.abs(dx_true))), "float32")
        weights_y = ops.cast(ops.exp(ops.mean(ops.abs(dy_true))), "float32")

        # Depth smoothness
        smoothness_x = dx_pred * weights_x
        smoothness_y = dy_pred * weights_y

        depth_smoothness_loss = ops.mean(abs(smoothness_x)) + ops.mean(
            abs(smoothness_y)
        )

        # Structural similarity (SSIM) index
        ssim_loss = ops.mean(
            1
            - tf.image.ssim(
                target, pred, max_val=WIDTH, filter_size=7, k1=0.01**2, k2=0.03**2
            )
        )
        # Point-wise depth
        l1_loss = ops.mean(ops.abs(target - pred))

        loss = (
            (self.ssim_loss_weight * ssim_loss)
            + (self.l1_loss_weight * l1_loss)
            + (self.edge_loss_weight * depth_smoothness_loss)
        )

        return loss, ssim_loss, l1_loss, depth_smoothness_loss

    @property
    def metrics(self):
        return [self.loss_metric, self.ssim_loss_metric, self.l1_loss_metric, self.edge_loss_metric]

    def train_step(self, batch_data):
        input, target = batch_data
        with tf.GradientTape() as tape:
            pred = self(input, training=True)
            loss, ssim_loss, l1_loss, edge_loss = self.calculate_loss(target, pred)

        gradients = tape.gradient(loss, self.trainable_variables)
        self.optimizer.apply_gradients(zip(gradients, self.trainable_variables))
        self.loss_metric.update_state(loss)
        self.ssim_loss_metric.update_state(ssim_loss)
        self.l1_loss_metric.update_state(l1_loss)
        self.edge_loss_metric.update_state(edge_loss)
        return {
            "loss": self.loss_metric.result(),
            "ssim_loss": self.ssim_loss_metric.result(),
            "l1_loss": self.l1_loss_metric.result(),
            "edge_loss": self.edge_loss_metric.result(),
        }

    def test_step(self, batch_data):
        input, target = batch_data

        pred = self(input, training=False)
        loss, ssim_loss, l1_loss, edge_loss = self.calculate_loss(target, pred)

        self.loss_metric.update_state(loss)
        self.ssim_loss_metric.update_state(ssim_loss)
        self.l1_loss_metric.update_state(l1_loss)
        self.edge_loss_metric.update_state(edge_loss)
        return {
            "loss": self.loss_metric.result(),
            "ssim_loss": self.ssim_loss_metric.result(),
            "l1_loss": self.l1_loss_metric.result(),
            "edge_loss": self.edge_loss_metric.result(),
        }

    def call(self, x):
        c1, p1 = self.downscale_blocks[0](x)
        c2, p2 = self.downscale_blocks[1](p1)
        c3, p3 = self.downscale_blocks[2](p2)
        c4, p4 = self.downscale_blocks[3](p3)

        bn = self.bottle_neck_block(p4)

        u1 = self.upscale_blocks[0](bn, c4)
        u2 = self.upscale_blocks[1](u1, c3)
        u3 = self.upscale_blocks[2](u2, c2)
        u4 = self.upscale_blocks[3](u3, c1)

        return self.conv_layer(u4)

> **Exercise 11-3:**
> 1. In one or a few sentences explain the importance of using a multi-scale backbone architecture for the depth estimation task. 
> 2. Explain the purpose of each of the loss function components in one or two sentences each.
> 3. In one or two sentences, explain why a multi-component loss function is required for this problem.    

> **Answers:**
> 1.         
> 2. By component the loss function components are as follows:  
>    a.        
>    b.        
>    c.            
> 3.           

## Model training

With the model and loss function defined, we are ready to train the model. Execute this code to train the model.  

In [ ]:
#optimizer = keras.optimizers.SGD(
optimizer = keras.optimizers.AdamW(
    learning_rate=LR,
    weight_decay=weight_decay,
#    nesterov=False,
)
model = DepthEstimationModel()
# Compile the model
model.compile(optimizer)

train_loader = DataGenerator(
    data=df[:260].reset_index(drop="true"), batch_size=BATCH_SIZE, dim=(HEIGHT, WIDTH)
)
validation_loader = DataGenerator(
    data=df[260:].reset_index(drop="true"), batch_size=BATCH_SIZE, dim=(HEIGHT, WIDTH)
)
history = model.fit(
    train_loader,
    epochs=EPOCHS,
    validation_data=validation_loader,
)

### Visualizing loss components

To evaluate the model training, execute the code in the cell below.  

In [ ]:
def plot_loss(history, loss_name, ax):
    ax.plot(history.history[loss_name], label=f'Training {loss_name}')
    ax.plot(history.history[f'val_{loss_name}'], label=f'Validation {loss_name}')
    ax.set_title(f'{loss_name.replace("_", " ").title()} over Epochs')
    ax.set_xlabel('Epoch')
    ax.set_ylabel(loss_name.replace("_", " ").title())
    ax.legend()
    ax.grid(True)

_,ax = plt.subplots(2,2, figsize=(10,10))
ax = ax.flatten()
loss_names = ['ssim_loss', 'l1_loss', 'edge_loss', 'loss']
for ax_i, loss_name in zip(ax, loss_names):
  plot_loss(history, loss_name, ax_i)
plt.show()

At this point one should be asking the question if this model is over-fit. If we only consider the training curves, the answer would have to be yes, the model is over-fitting. But, at the same time the validation curves for the three loss components as well as the regularized total loss are all fairly smooth. Further, experimentation with weight decay, learning rate and weights for the components of the loss function showed that the values used to be close to optimal. Therefore, we can conclude that while the fit of the model is not idea, it is most likely not seriously over-fit.        

> **Exercise 11-4:** Based on the training curves shown and the discussion above, answer these questions in one or a few sentences.
> 1. Notice that while the training edge loss decreases following a fairly smooth curve, the validation edge loss is nearly constant. Given the properties of depth maps and their gradients inherent in a depth map, how can you explain this outcome? *Hint:* You should have a look at the difference between the code for`train_step` and `test_step` methods in the model.     
> 2. Given that the loss function has three components, how can you explain the relatively smooth validation loss curves resulting from training with more erratic loss curves?    

> **Answers:**
> 1.       
> 2.      

## Visualizing model output

We visualize the model output over the validation set.
The first image is the RGB image, the second image is the ground truth depth map image
and the third one is the predicted depth map image.

In [ ]:
test_loader = next(
    iter(
        DataGenerator(
            data=df[265:].reset_index(drop="true"), batch_size=6, dim=(HEIGHT, WIDTH)
        )
    )
)
visualize_depth_map(test_loader, test=True, model=model)

test_loader = next(
    iter(
        DataGenerator(
            data=df[300:].reset_index(drop="true"), batch_size=6, dim=(HEIGHT, WIDTH)
        )
    )
)
visualize_depth_map(test_loader, test=True, model=model)

It is clear that the depth maps generated do not match the target particularly well. There is clearly a systematic bias where the estimated distances are consistently too great. This outcome is not unusual for simple monocular depth map generation model. 

While the actual depths estimated are biased, the depth ordering of the scenes are generally correct.  

> **Exercise 11-5:** A number of the examples shown above have aspects that are challenging for any depth estimation model. In one or a few sentences discuss the following difficulties with the examples.      
> 1. The depth map of the first image is nearly featureless. What properties of the original image lead to this results?    
> 2. The third image from the top has highly variable illumination from the lamp in the room. How has this illumination affected the depth estimate generated.    
> 3. The images showing the auditorium seats exhibit a property that commonly causes difficulty for depth estimation algorithms. What is this property and why is it problematic.         

> **Answers:**
> 1.        
> 2.         
> 3.       

#### Copyright 2026, Stephen F. Elston. All rights reserved.   